In [ ]:
import sys
from pathlib import Path

for project_root in (Path.cwd(), *Path.cwd().parents):
    if (project_root / "src").exists():
        sys.path.insert(0, str(project_root))
        break

import pandas as pd
from src.data.loaders import (load_yambda, split_and_reindex)

In [4]:
config = {
    "yambda-retrieval": {
        "max_seq_len": 100,
        "test_quantile": 0.1,
        "interactions_path": "/kaggle/input/datasets/andrewkhoroshilov/datasets-action-speak-louder-then-words/yambda-10m.parquet",
        "col_mapping": {
            "uid": "user_id",
            "item_id": "item_id",
            "timestamp": "timestamp",
        },
    },
}

In [5]:
yambda_df = load_yambda(config["yambda-retrieval"]["interactions_path"], config=config["yambda-retrieval"])

In [6]:
yambda_train, yambda_test, yambda_data_description = split_and_reindex(yambda_df, config=config["yambda-retrieval"])

In [7]:
datasets = {
    "yambda": {
        "train": yambda_train,
        "test": yambda_test,
        "desc": yambda_data_description,
    }
}

In [8]:
# статистики по датасетам

stats = []
for name, d in datasets.items():
    train, test, desc = d["train"], d["test"], d["desc"]
    n_train = len(train)
    n_test = len(test)
    n_total = n_train + n_test
    stats.append({
        "dataset": name,
        "n_train": n_train,
        "n_test": n_test,
        "n_total": n_total,
        "train_share": round(n_train / n_total, 3),
        "test_share": round(n_test  / n_total, 3),
        "n_train_users": train[desc["users"]].nunique(),
        "n_test_users": test[desc["users"]].nunique(),
        "n_items": train[desc["items"]].nunique(),
    })

pd.DataFrame(stats)

,dataset,n_train,n_test,n_total,train_share,test_share,n_train_users,n_test_users,n_items
0,yambda,7473676,748537,8222213,0.909,0.091,79059,53927,163448


In [ ]:
import torch

from src.HSTU.training.hstu import (
    HSTUExperimentConfig,
    build_hstu_dataloaders,
    evaluate_hstu,
    train_hstu,
)
from src.HSTU.models.hstu import HSTUModel


## HSTU Yambda experiment

Запуск HSTU-пайплайна на Yambda

In [20]:
base_hstu_config = HSTUExperimentConfig(
    max_seq_len=config["yambda-retrieval"]["max_seq_len"],
    train_batch_size=128,
    eval_batch_size=128,
    num_epochs=10,
    learning_rate=1e-3,
    weight_decay=0.0,
    topk=100,
    embedding_dim=64,
    num_blocks=2,
    num_heads=2,
    linear_dim=64,
    attention_dim=64,
    input_dropout_rate=0.2,
    linear_dropout_rate=0.2,
    num_negatives=512,
    sampling_strategy="local",
    softmax_temperature=0.05,
    user_embedding_norm="l2_norm",
    enable_relative_attention_bias=True,
    l2_norm_embeddings=True,
    l2_norm_eps=1e-6,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

sl_hstu_config = HSTUExperimentConfig(
    **{
        **base_hstu_config.__dict__,
        "stochastic_length_alpha": 1.5,
        "stochastic_length_sampling": "random_subsequence",
    }
)

baseline_train_loader, baseline_eval_loader, baseline_train_dataset, baseline_eval_dataset, baseline_targets = build_hstu_dataloaders(
    train=yambda_train,
    test=yambda_test,
    max_seq_len=base_hstu_config.max_seq_len,
    train_batch_size=base_hstu_config.train_batch_size,
    eval_batch_size=base_hstu_config.eval_batch_size,
    user_col=yambda_data_description["users"],
    item_col=yambda_data_description["items"],
    time_col=yambda_data_description["timestamp"],
    num_workers=base_hstu_config.num_workers,
)

sl_train_loader, sl_eval_loader, sl_train_dataset, sl_eval_dataset, sl_targets = build_hstu_dataloaders(
    train=yambda_train,
    test=yambda_test,
    max_seq_len=sl_hstu_config.max_seq_len,
    train_batch_size=sl_hstu_config.train_batch_size,
    eval_batch_size=sl_hstu_config.eval_batch_size,
    user_col=yambda_data_description["users"],
    item_col=yambda_data_description["items"],
    time_col=yambda_data_description["timestamp"],
    num_workers=sl_hstu_config.num_workers,
    stochastic_length_alpha=sl_hstu_config.stochastic_length_alpha,
    stochastic_length_sampling=sl_hstu_config.stochastic_length_sampling,
)

hstu_config = sl_hstu_config
hstu_train_loader = sl_train_loader
hstu_eval_loader = sl_eval_loader
hstu_train_dataset = sl_train_dataset
hstu_eval_dataset = sl_eval_dataset
hstu_targets = sl_targets

{
    "baseline": (len(baseline_train_dataset), len(baseline_eval_dataset), len(baseline_targets)),
    "stochastic_length": (len(sl_train_dataset), len(sl_eval_dataset), len(sl_targets)),
    "sl_alpha": sl_hstu_config.stochastic_length_alpha,
    "sl_sampling": sl_hstu_config.stochastic_length_sampling,
}


{'baseline': (121756, 53927, 53927),
 'stochastic_length': (73491, 53927, 53927),
 'sl_alpha': 1.5,
 'sl_sampling': 'random_subsequence'}

In [21]:
def build_hstu_model_and_optimizer(experiment_config):
    model = HSTUModel(
        num_items=yambda_data_description["n_items"],
        embedding_dim=experiment_config.embedding_dim,
        max_seq_len=experiment_config.max_seq_len,
        num_blocks=experiment_config.num_blocks,
        num_heads=experiment_config.num_heads,
        linear_dim=experiment_config.linear_dim,
        attention_dim=experiment_config.attention_dim,
        num_negatives=experiment_config.num_negatives,
        softmax_temperature=experiment_config.softmax_temperature,
        sampling_strategy=experiment_config.sampling_strategy,
        user_embedding_norm=experiment_config.user_embedding_norm,
        l2_norm_embeddings=experiment_config.l2_norm_embeddings,
        l2_norm_eps=experiment_config.l2_norm_eps,
        item_id_offset=experiment_config.item_id_offset,
        input_dropout_rate=experiment_config.input_dropout_rate,
        linear_dropout_rate=experiment_config.linear_dropout_rate,
        attn_dropout_rate=experiment_config.attn_dropout_rate,
        enable_relative_attention_bias=experiment_config.enable_relative_attention_bias,
        relative_attention_num_buckets=experiment_config.relative_attention_num_buckets,
    )
    model.to(experiment_config.device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=experiment_config.learning_rate,
        weight_decay=experiment_config.weight_decay,
    )
    return model, optimizer


baseline_model, baseline_optimizer = build_hstu_model_and_optimizer(base_hstu_config)
sl_model, sl_optimizer = build_hstu_model_and_optimizer(sl_hstu_config)

hstu_model = sl_model
hstu_optimizer = sl_optimizer


In [22]:
params_sum = 0
trainable_sum = 0

for name, module in sl_model.encoder.named_children():
    params = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    print(f"{name}: total={params:,}, trainable={trainable:,}")

    params_sum += params
    trainable_sum += trainable
print()
print(f"params_sum={params_sum:,}, trainable_sum={trainable_sum:,}")


item_embeddings: total=10,460,736, trainable=10,460,736
pos_embeddings: total=6,400, trainable=6,400
input_dropout: total=0, trainable=0
blocks: total=83,472, trainable=83,472

params_sum=10,550,608, trainable_sum=10,550,608


In [23]:
def train_hstu_with_epoch_timing(
    model,
    optimizer,
    train_loader,
    eval_loader,
    targets,
    experiment_config,
    run_name,
):
    losses = []
    eval_history = []
    epoch_times = []

    for epoch in range(experiment_config.num_epochs):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start_time = time.perf_counter()

        epoch_losses = train_hstu(
            model=model,
            train_loader=train_loader,
            optimizer=optimizer,
            num_epochs=1,
            device=experiment_config.device,
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        epoch_time = time.perf_counter() - start_time

        epoch_metrics, _ = evaluate_hstu(
            model=model,
            eval_loader=eval_loader,
            targets=targets,
            catalog_size=yambda_data_description["n_items"],
            topk=experiment_config.topk,
            device=experiment_config.device,
            filter_seen=experiment_config.filter_seen,
        )

        losses.extend(epoch_losses)
        eval_history.append(epoch_metrics)
        epoch_times.append(epoch_time)
        print(
            f"{run_name} epoch {epoch + 1}/{experiment_config.num_epochs}: "
            f"train_time={epoch_time:.2f}s, "
            f"loss={epoch_losses[-1]:.4f}"
        )

    return losses, eval_history, epoch_times


baseline_losses, baseline_eval_history, baseline_epoch_times = train_hstu_with_epoch_timing(
    model=baseline_model,
    optimizer=baseline_optimizer,
    train_loader=baseline_train_loader,
    eval_loader=baseline_eval_loader,
    targets=baseline_targets,
    experiment_config=base_hstu_config,
    run_name="baseline",
)

sl_losses, sl_eval_history, sl_epoch_times = train_hstu_with_epoch_timing(
    model=sl_model,
    optimizer=sl_optimizer,
    train_loader=sl_train_loader,
    eval_loader=sl_eval_loader,
    targets=sl_targets,
    experiment_config=sl_hstu_config,
    run_name="stochastic_length",
)

{
    "baseline_losses": baseline_losses,
    "baseline_epoch_times": baseline_epoch_times,
    "baseline_eval_history": baseline_eval_history,
    "stochastic_length_losses": sl_losses,
    "stochastic_length_epoch_times": sl_epoch_times,
    "stochastic_length_eval_history": sl_eval_history,
}


epoch 1/1:   0%|          | 0/951 [00:00<?, ?it/s]

HSTU evaluation:   0%|          | 0/422 [00:00<?, ?it/s]

baseline epoch 1/10: train_time=184.27s, loss=4.8072


epoch 1/1:   0%|          | 0/951 [00:00<?, ?it/s]

HSTU evaluation:   0%|          | 0/422 [00:00<?, ?it/s]

...

stochastic_length epoch 9/10: train_time=97.50s, loss=2.5977


epoch 1/1:   0%|          | 0/574 [00:00<?, ?it/s]

HSTU evaluation:   0%|          | 0/422 [00:00<?, ?it/s]

stochastic_length epoch 10/10: train_time=97.50s, loss=2.5529


{'baseline_losses': [4.807200543637281,
  3.452825750838319,
  3.0535567007355633,
  2.8627131947959636,
  2.735822655801141,
  2.647776392356078,
  2.5851109612000602,
  2.537566627989808,
  2.500522322208473,
  2.470277765947938],
 'baseline_epoch_times': [184.26849842900015,
  183.8988150990001,
  183.8198922270003,
  183.9355932819999,
  183.7706042640002,
  183.9441452320002,
  183.80127354600017,
  183.95441094999978,
  183.77082741599997,
  184.08849633900036],
 'baseline_eval_history': [{'hitrate': 0.3561481261705639,
   'recall': 0.07233678162323982,
   'ndcg': 0.033136276941962674,
   'coverage': 0.10615608633938622},
  {'hitrate': 0.4557457303391622,
   'recall': 0.10558510040727226,
   'ndcg': 0.050972129309897266,
   'coverage': 0.2978133718369145},
  {'hitrate': 0.4755502809353385,
   'recall': 0.11449449576557533,
   'ndcg': 0.05498467192892031,
   'coverage': 0.38592702266164164},
  {'hitrate': 0.48404324364418566,
   'recall': 0.11791118698593223,
   'ndcg': 0.05655095

In [25]:
training_rows = []
for run_name, run_losses, run_epoch_times, run_eval_history in (
    ("baseline", baseline_losses, baseline_epoch_times, baseline_eval_history),
    ("stochastic_length", sl_losses, sl_epoch_times, sl_eval_history),
):
    for epoch, (loss, train_time, metrics) in enumerate(
        zip(run_losses, run_epoch_times, run_eval_history),
        start=1,
    ):
        training_rows.append(
            {
                "run": run_name,
                "epoch": epoch,
                "loss": loss,
                "train_time_sec": train_time,
                **metrics,
            }
        )

training_summary = pd.DataFrame(training_rows)
training_summary

,run,epoch,loss,train_time_sec,hitrate,recall,ndcg,coverage
0,baseline,1,4.807201,184.268498,0.356148,0.072337,0.033136,0.106156
1,baseline,2,3.452826,183.898815,0.455746,0.105585,0.050972,0.297813
2,baseline,3,3.053557,183.819892,0.475550,0.114494,0.054985,0.385927
3,baseline,4,2.862713,183.935593,0.484043,0.117911,0.056551,0.468773
4,baseline,5,2.735823,183.770604,0.495485,0.123532,0.059688,0.516348
5,baseline,6,2.647776,183.944145,0.495058,0.123772,0.059876,0.536238
6,baseline,7,2.585111,183.801274,0.497895,0.125178,0.060571,0.565489
7,baseline,8,2.537567,183.954411,0.499657,0.125635,0.060898,0.581316
8,baseline,9,2.500522,183.770827,0.503792,0.126725,0.061739,0.584082
9,baseline,10,2.470278,184.088496,0.502420,0.126344,0.061444,0.591344
